# 采购增量分析

读取按抓取日归档的采购入库底表，逐步核对范围、指标和结论。
口径见 [采购增量分析](../docs/采购增量分析.md)。先在项目环境安装 `pip install -e ".[notebooks]"`，并使用同一环境的 Jupyter 内核。


In [ ]:
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.styles import Border, Font, PatternFill, Side
from openpyxl.styles import Alignment

np.set_printoptions(suppress=True)

In [ ]:
from pipeline import load_source
from paths import OUTPUTS_ROOT

snapshot_date = None  # 指定 YYYY-MM-DD 可复用历史归档；None 取最新一版
df, data_meta = load_source("procurement-inbound", snapshot_date=snapshot_date)

# 获取数据基本信息
# df.info()

In [ ]:
# 分析窗口，单位：月
analysis_window_months = 12
# 筛选条件，最近多少个月内采购金额超过多少万
recent_filter_months = 6   
min_recent_amount_wan = 5.0

In [ ]:
target_boards = ["检验", "检验产品", "诊断"]

analysis_cols = [
    "板块",
    "物料编码",
    "物料名称",
    "规格",
    "主单位",
    "品牌",
    "主品牌",
    "一级分类",
    "二级分类",
    "三级分类",
    "四级分类",
    "五级分类",
    "设备试剂分类",
    "进口&国产",
    "供应商名称",
    "采购入库年份",
    "采购入库月份",
    "数量合计",
    "采购不含税金额(万)",
    "采购不含税单价",
]

available_cols = [col for col in analysis_cols if col in df.columns]
source_df = df.loc[df["板块"].isin(target_boards), available_cols].copy()

source_df = source_df.rename(columns={
    "采购入库年份": "年份",
    "采购入库月份": "月份",
    "数量合计": "数量",
})

for col in ["年份", "月份", "物料编码"]:
    if col in source_df.columns:
        source_df[col] = pd.to_numeric(source_df[col], errors="coerce")

for col in ["数量", "采购不含税金额(万)", "采购不含税单价"]:
    if col in source_df.columns:
        source_df[col] = pd.to_numeric(source_df[col], errors="coerce")

source_df = source_df.dropna(subset=["年份", "月份", "物料编码", "数量"]).copy()
source_df["年份"] = source_df["年份"].astype(int)
source_df["月份"] = source_df["月份"].astype(int)
source_df["年月"] = pd.to_datetime(
    source_df["年份"].astype(str) + "-" + source_df["月份"].astype(str).str.zfill(2) + "-01"
)

report_end_month = source_df["年月"].max()
window_start_month = report_end_month - pd.DateOffset(months=analysis_window_months - 1)
analysis_window_df = source_df.loc[source_df["年月"].between(window_start_month, report_end_month)].copy()

recent_amount_by_material = (
    analysis_window_df.loc[
        analysis_window_df["年月"] >= (report_end_month - pd.DateOffset(months=recent_filter_months - 1)),
        ["物料编码", "采购不含税金额(万)"],
    ]
    .groupby("物料编码", as_index=False, observed=True)["采购不含税金额(万)"]
    .sum()
    .rename(columns={"采购不含税金额(万)": "最近6个月累计采购不含税金额(万)"})
)

eligible_materials = recent_amount_by_material.loc[
    recent_amount_by_material["最近6个月累计采购不含税金额(万)"] > min_recent_amount_wan,
    "物料编码",
]

analysis_df = analysis_window_df.loc[analysis_window_df["物料编码"].isin(eligible_materials)].copy()

print(f"完成最近 {analysis_window_months} 个月分析窗口与最近 {recent_filter_months} 个月金额过滤。")
print(f"窗口起始月份: {window_start_month:%Y-%m}, 结束月份: {report_end_month:%Y-%m}")
print(f"通过金额过滤的物料数: {analysis_df['物料编码'].nunique()}")
print(f"窗口内过滤后明细行数: {len(analysis_df)}")
print("完成源数据清洗与板块过滤，后续仅使用分析窗口内的数据。")


In [ ]:
material_dim_cols = [
    col
    for col in [
        "物料名称",
        "规格",
        "主单位",
        "品牌",
        "主品牌",
        "一级分类",
        "二级分类",
        "三级分类",
        "四级分类",
        "五级分类",
        "设备试剂分类",
        "进口&国产",
    ]
    if col in analysis_df.columns
]

material_dim = (
    analysis_df.sort_values(["物料编码", "年月"])
    .groupby("物料编码", observed=True)[material_dim_cols]
    .first()
    .reset_index()
)

month_index = pd.DataFrame({"年月": pd.date_range(window_start_month, report_end_month, freq="MS")})
monthly = material_dim.assign(_merge_key=1).merge(month_index.assign(_merge_key=1), on="_merge_key").drop(columns="_merge_key")

monthly_amount = (
    analysis_df.groupby(["物料编码", "年月"], as_index=False, observed=True)
    .agg({"数量": "sum", "采购不含税金额(万)": "sum"})
)
monthly = monthly.merge(monthly_amount, on=["物料编码", "年月"], how="left")
monthly["数量"] = monthly["数量"].fillna(0.0)
monthly["采购不含税金额(万)"] = monthly["采购不含税金额(万)"].fillna(0.0)
monthly["月加权平均价"] = monthly["采购不含税金额(万)"] / monthly["数量"].replace(0, np.nan)

print("完成物料月度汇总，已补齐分析窗口内的零采购月份。")


In [ ]:
def calc_positive_streak(growth_series: pd.Series) -> pd.Series:
    streak = 0
    streak_values = []
    for growth_value in growth_series:
        if pd.notna(growth_value) and growth_value > 0:
            streak += 1
        else:
            streak = 0
        streak_values.append(streak)
    return pd.Series(streak_values, index=growth_series.index)


def add_monthly_metrics(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("年月").copy()
    quantity = group["数量"]
    amount = group["采购不含税金额(万)"]

    previous_quantity = quantity.shift(1)
    growth = quantity.pct_change()
    growth = growth.where(previous_quantity > 0, np.nan)

    group["最近1个月环比增长率"] = growth
    group["最近3个月平均采购量"] = quantity.rolling(3, min_periods=3).mean()
    group["前3个月平均采购量"] = quantity.shift(3).rolling(3, min_periods=3).mean()
    group["采购量趋势指数"] = group["最近3个月平均采购量"] / group["前3个月平均采购量"] - 1
    group["连续增长月数"] = calc_positive_streak(growth)
    group["最近3个月采购量占比"] = quantity.rolling(3, min_periods=3).sum() / quantity.rolling(12, min_periods=12).sum()
    group["最近12个月累计采购不含税金额(万)"] = amount.rolling(12, min_periods=12).sum()
    group["最近3个月零采购月份"] = (quantity == 0).rolling(3, min_periods=3).sum()
    group["月加权平均价(元)"] = amount * 10000 / quantity.replace(0, np.nan)
    return group


monthly_metric_frames = []
# 逐物料计算连续增长、趋势和占比指标：这里的 monthly 已经补齐了分析窗口内的零采购月份。
for _, group in monthly.groupby("物料编码", sort=False, observed=True):
    monthly_metric_frames.append(add_monthly_metrics(group))
monthly = pd.concat(monthly_metric_frames, ignore_index=True)

# 供应商维度的金额汇总，用于识别该物料的核心供应商，以及核心供应商在总金额中的占比。
supplier_amount = (
    analysis_df.groupby(["物料编码", "供应商名称"], as_index=False, observed=True)["采购不含税金额(万)"]
    .sum()
    .sort_values(["物料编码", "采购不含税金额(万)"], ascending=[True, False])
)
# 核心供应商名称取该物料在分析窗口内累计采购不含税金额最高的供应商。
core_supplier = supplier_amount.drop_duplicates("物料编码").rename(
    columns={"供应商名称": "核心供应商名称", "采购不含税金额(万)": "核心供应商金额(万)"}
)
# 先汇总物料在分析窗口内的总金额，再算核心供应商供货占比。
supplier_total = (
    analysis_df.groupby("物料编码", as_index=False, observed=True)["采购不含税金额(万)"]
    .sum()
    .rename(columns={"采购不含税金额(万)": "最近12个月累计采购不含税金额(万)"})
)
core_supplier = core_supplier.merge(supplier_total, on="物料编码", how="left")
core_supplier["单一供应商供货占比"] = core_supplier["核心供应商金额(万)"] / core_supplier["最近12个月累计采购不含税金额(万)"]

summary = pd.DataFrame(
    {
        "指标": ["过滤后明细行数", "月度汇总行数", "覆盖物料数", "核心供应商覆盖物料数"],
        "数值": [
            len(analysis_df),
            len(monthly),
            monthly["物料编码"].nunique(),
            core_supplier["物料编码"].nunique(),
        ],
    }
)

print("完成物料增量指标计算。")

In [ ]:
latest = (
    monthly.sort_values(["物料编码", "年月"])
    .groupby("物料编码", as_index=False)
    .tail(1)
    .merge(
        core_supplier[["物料编码", "核心供应商名称", "单一供应商供货占比"]],
        on="物料编码",
        how="left",
    )
)

latest = latest.loc[latest["最近3个月零采购月份"] == 0].copy()




# 依据文档里的三档规则，将每个物料映射到中文优先级。
def classify_priority(row: pd.Series) -> str:
    core_condition = (
        row["采购量趋势指数"] >= 0.2
        and ((row["最近1个月环比增长率"] >= 0.1) or (row["连续增长月数"] >= 2))
        and row["最近3个月零采购月份"] == 0
    )
    if core_condition:
        return "优先谈判"
    if row["采购量趋势指数"] >= 0.1 or row["最近3个月采购量占比"] >= 0.3:
        return "观察"
    return "不建议"


priority_order = {"优先谈判": 0, "观察": 1, "不建议": 2}
latest["谈判优先级"] = latest.apply(classify_priority, axis=1)
latest["谈判优先级排序"] = latest["谈判优先级"].map(priority_order)
latest = latest.sort_values(
    ["谈判优先级排序", "采购量趋势指数", "最近3个月采购量占比"],
    ascending=[True, False, False],
).drop(columns=["谈判优先级排序"]).reset_index(drop=True)

report_df = latest[
    [
        "物料编码",
        "物料名称",
        "规格",
        "品牌",
        "五级分类",
        "核心供应商名称",
        "最近3个月平均采购量",
        "前3个月平均采购量",
        "采购量趋势指数",
        "最近1个月环比增长率",
        "连续增长月数",
        "最近3个月采购量占比",
        "最近12个月累计采购不含税金额(万)",
        "单一供应商供货占比",
        "谈判优先级",
    ]
].copy()

# 展示层把增长类指标格式化成百分比。
def format_growth_display(value):
    if pd.isna(value):
        return ""
    if np.isinf(value):
        return "起量" if value > 0 else "无基数"
    return f"{value:.2%}"

report_display_df = report_df.copy()
for col in ["采购量趋势指数", "最近1个月环比增长率", "最近3个月采购量占比", "单一供应商供货占比"]:
    if col in report_display_df.columns:
        report_display_df[col] = report_display_df[col].map(format_growth_display)

report_dir = OUTPUTS_ROOT / "reports" / data_meta["snapshot_date"]
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / f"{report_end_month:%Y-%m}物料增量分析报告.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    report_df.to_excel(writer, sheet_name="报告", index=False)
    report_sheet = writer.sheets["报告"]
    report_sheet.freeze_panes = "A2"
    report_sheet.row_dimensions[1].height = 36
    header_display_map = {
        "最近3个月平均采购量": "最近3个月\n平均采购量",
        "前3个月平均采购量": "前3个月\n平均采购量",
        "采购量趋势指数": "采购量趋势\n指数",
        "最近1个月环比增长率": "最近1个月\n环比增长率",
        "连续增长月数": "连续增长\n月数",
        "最近3个月采购量占比": "最近3个月\n采购量占比",
        "最近12个月累计采购不含税金额(万)": "最近12个月累计采购\n不含税金额(万)",
        "单一供应商供货占比": "单一供应商\n供货占比",
    }
    header_width_map = {
        "最近3个月平均采购量": 16,
        "前3个月平均采购量": 16,
        "采购量趋势指数": 14,
        "最近1个月环比增长率": 18,
        "连续增长月数": 12,
        "最近3个月采购量占比": 16,
        "最近12个月累计采购不含税金额(万)": 20,
        "单一供应商供货占比": 16,
        "谈判优先级": 12,
    }
    for header_cell in report_sheet[1]:
        original_header = header_cell.value
        if original_header in header_display_map:
            header_cell.value = header_display_map[original_header]
        header_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        report_sheet.column_dimensions[header_cell.column_letter].width = header_width_map.get(original_header, 12)
    report_column_index = {column: idx + 1 for idx, column in enumerate(report_df.columns)}
    for column in ["最近3个月平均采购量", "前3个月平均采购量", "最近12个月累计采购不含税金额(万)"]:
        if column in report_column_index:
            for cell in report_sheet.iter_cols(min_col=report_column_index[column], max_col=report_column_index[column], min_row=2):
                for item in cell:
                    item.number_format = "0.00"
    for column in ["采购量趋势指数", "最近1个月环比增长率", "最近3个月采购量占比", "单一供应商供货占比"]:
        if column in report_column_index:
            for cell in report_sheet.iter_cols(min_col=report_column_index[column], max_col=report_column_index[column], min_row=2):
                for item in cell:
                    item.number_format = "0.00%"
    monthly_export_df = monthly.copy()
    monthly_export_df.to_excel(writer, sheet_name="月度数据", index=False)

    rule_rows = [
        ("报表用途", "识别采购量持续增长且具备谈判价值的物料，形成月度谈判候选清单。"),
        ("分析范围", "仅覆盖检验、检验产品、诊断三个板块；以源数据最大采购年月为结束月份，向前取12个月。"),
        ("金额准入", "最近6个月累计采购不含税金额大于5万元的物料才进入后续分析。"),
        ("连续采购准入", "最近3个月存在零采购月份的物料直接排除，不参与优先级分类。"),
        ("优先谈判", "采购量趋势指数大于等于20%，且最近1个月环比增长率大于等于10%或连续增长月数大于等于2。表示采购量已明显提升且近期增长仍在延续，应优先核实需求原因并准备谈判。"),
        ("观察", "未达到优先谈判条件，但采购量趋势指数大于等于10%或最近3个月采购量占比大于等于30%。表示已出现增长信号或近期需求集中，建议按月跟踪。"),
        ("不建议", "不满足优先谈判或观察条件。当前采购量增长信号不足，暂不建议因采购增量启动谈判。"),
        ("最近1个月环比增长率", "(本月采购量-上月采购量)/上月采购量；仅当上月采购量大于0时计算。"),
        ("采购量趋势指数", "最近3个月平均采购量/前3个月平均采购量-1，用于判断近期采购量是否整体上台阶。"),
        ("连续增长月数", "截至当前月，采购量环比连续为正的月份数。"),
        ("最近3个月采购量占比", "最近3个月采购总量/最近12个月采购总量，用于判断需求是否集中在近期释放。"),
        ("使用提示", "报表提供采购增量证据和优先级，不代替对一次性备货、项目采购、价格、替代供应商和质量表现的业务判断。"),
    ]
    rule_df = pd.DataFrame(rule_rows, columns=["模块", "规则说明"])
    rule_df.to_excel(writer, sheet_name="规则说明", index=False)
    rule_sheet = writer.sheets["规则说明"]
    rule_sheet.freeze_panes = "A2"
    rule_sheet.column_dimensions["A"].width = 24
    rule_sheet.column_dimensions["B"].width = 100
    for header_cell in rule_sheet[1]:
        header_cell.alignment = Alignment(horizontal="center", vertical="center")
    for row in rule_sheet.iter_rows(min_row=2):
        row[0].alignment = Alignment(vertical="top", wrap_text=True)
        row[1].alignment = Alignment(vertical="top", wrap_text=True)
        rule_sheet.row_dimensions[row[0].row].height = 42

summary = pd.DataFrame(
    {
        "指标": ["过滤后明细行数", "月度汇总行数", "覆盖物料数", "报告物料数", "优先谈判数", "观察数", "不建议数"],
        "数值": [
            len(analysis_df),
            len(monthly),
            monthly["物料编码"].nunique(),
            len(report_df),
            int((report_df["谈判优先级"] == "优先谈判").sum()),
            int((report_df["谈判优先级"] == "观察").sum()),
            int((report_df["谈判优先级"] == "不建议").sum()),
        ],
    }
)

print("已完成物料维度汇总分析，并导出 Excel 报告。")
print(f"报告路径: {report_path}")


In [ ]:
# 图表页聚焦两项决策：各优先级覆盖规模，以及优先谈判物料的金额排名。
workbook = load_workbook(report_path)
if "图表" in workbook.sheetnames:
    del workbook["图表"]

chart_sheet = workbook.create_sheet("图表")
chart_sheet.sheet_view.showGridLines = False
navy_fill = PatternFill("solid", fgColor="1F4E78")
light_blue_fill = PatternFill("solid", fgColor="D9EAF7")
pale_blue_fill = PatternFill("solid", fgColor="EAF2F8")
yellow_fill = PatternFill("solid", fgColor="FFF2CC")
thin_side = Side(style="thin", color="D9E2F3")
table_border = Border(left=thin_side, right=thin_side, top=thin_side, bottom=thin_side)

chart_sheet.merge_cells("A1:H1")
chart_sheet["A1"] = "采购增量谈判概览"
chart_sheet["A1"].fill = navy_fill
chart_sheet["A1"].font = Font(bold=True, color="FFFFFF", size=16)
chart_sheet["A1"].alignment = Alignment(horizontal="left", vertical="center")
chart_sheet.row_dimensions[1].height = 30
chart_sheet.merge_cells("A2:H2")
chart_sheet["A2"] = "用于判断谈判候选覆盖的采购规模，并优先锁定金额最高的优先谈判物料。金额单位：万元。"
chart_sheet["A2"].fill = light_blue_fill
chart_sheet["A2"].alignment = Alignment(vertical="center", wrap_text=True)
chart_sheet.row_dimensions[2].height = 28

for label_range, label in [("A4:B4", "优先谈判候选数"), ("D4:E4", "优先谈判覆盖金额（万元）")]:
    chart_sheet.merge_cells(label_range)
    cell = chart_sheet[label_range.split(":")[0]]
    cell.value = label
    cell.fill = pale_blue_fill
    cell.font = Font(bold=True, color="1F4E78", size=10)
    cell.alignment = Alignment(horizontal="center", vertical="center")

chart_sheet.merge_cells("A5:B6")
chart_sheet["A5"] = int((report_df["谈判优先级"] == "优先谈判").sum())
chart_sheet.merge_cells("D5:E6")
chart_sheet["D5"] = float(report_df.loc[report_df["谈判优先级"] == "优先谈判", "最近12个月累计采购不含税金额(万)"].sum())
for cell_ref in ["A5", "D5"]:
    cell = chart_sheet[cell_ref]
    cell.font = Font(bold=True, color="1F4E78", size=18)
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = table_border
chart_sheet["A5"].number_format = "#,##0"
chart_sheet["D5"].number_format = "#,##0.0"

priority_summary = (
    report_df.groupby("谈判优先级", observed=True)
    .agg(物料数=("物料编码", "size"), 采购金额万元=("最近12个月累计采购不含税金额(万)", "sum"))
    .reindex(["优先谈判", "观察", "不建议"])
    .fillna(0)
    .reset_index()
)
for col_index, value in enumerate(["谈判优先级", "物料数", "采购金额(万元)"], start=10):
    chart_sheet.cell(2, col_index, value)
for row_index, row in priority_summary.iterrows():
    excel_row = row_index + 3
    chart_sheet.cell(excel_row, 10, row["谈判优先级"])
    chart_sheet.cell(excel_row, 11, int(row["物料数"]))
    chart_sheet.cell(excel_row, 12, float(row["采购金额万元"]))

top_materials = (
    report_df.loc[report_df["谈判优先级"] == "优先谈判"]
    .nlargest(15, "最近12个月累计采购不含税金额(万)")
    .reset_index(drop=True)
)
top_headers = ["排名", "物料名称", "核心供应商", "采购金额(万元)", "图表标签"]
for col_index, value in enumerate(top_headers, start=10):
    chart_sheet.cell(8, col_index, value)
for row_index, row in top_materials.iterrows():
    excel_row = row_index + 9
    material_name = str(row["物料名称"])
    short_name = material_name[:18] + ("..." if len(material_name) > 18 else "")
    chart_sheet.cell(excel_row, 10, row_index + 1)
    chart_sheet.cell(excel_row, 11, material_name)
    chart_sheet.cell(excel_row, 12, row["核心供应商名称"])
    chart_sheet.cell(excel_row, 13, float(row["最近12个月累计采购不含税金额(万)"]))
    chart_sheet.cell(excel_row, 14, f"Top {row_index + 1}  {short_name}")

for row in chart_sheet.iter_rows(min_row=2, max_row=5, min_col=10, max_col=12):
    for cell in row:
        cell.border = table_border
        cell.alignment = Alignment(vertical="center")
for row in chart_sheet.iter_rows(min_row=8, max_row=23, min_col=10, max_col=14):
    for cell in row:
        cell.border = table_border
        cell.alignment = Alignment(vertical="center", wrap_text=True)
for header_range in ["J2:L2", "J8:N8"]:
    for cell in chart_sheet[header_range][0]:
        cell.fill = navy_fill
        cell.font = Font(bold=True, color="FFFFFF")
        cell.alignment = Alignment(horizontal="center", vertical="center")
for row_index in range(3, 6):
    chart_sheet.cell(row_index, 12).number_format = "#,##0.0"
for row_index in range(9, 24):
    chart_sheet.cell(row_index, 13).number_format = "#,##0.0"
    chart_sheet.row_dimensions[row_index].height = 30

priority_chart = BarChart()
priority_chart.type = "col"
priority_chart.style = 10
priority_chart.title = "各优先级覆盖采购金额（万元）"
priority_chart.y_axis.title = "采购金额（万元）"
priority_chart.height = 10
priority_chart.width = 18
priority_data = Reference(chart_sheet, min_col=12, min_row=2, max_row=5)
priority_categories = Reference(chart_sheet, min_col=10, min_row=3, max_row=5)
priority_chart.add_data(priority_data, titles_from_data=True)
priority_chart.set_categories(priority_categories)
priority_chart.legend = None
priority_chart.dataLabels = DataLabelList()
priority_chart.dataLabels.showVal = True
chart_sheet.add_chart(priority_chart, "A8")

top_chart = BarChart()
top_chart.type = "bar"
top_chart.style = 10
top_chart.title = "优先谈判物料 Top 15（最近12个月采购金额，万元）"
top_chart.x_axis.title = "采购金额（万元）"
top_chart.height = 18
top_chart.width = 18
top_data = Reference(chart_sheet, min_col=13, min_row=8, max_row=23)
top_categories = Reference(chart_sheet, min_col=14, min_row=9, max_row=23)
top_chart.add_data(top_data, titles_from_data=True)
top_chart.set_categories(top_categories)
top_chart.legend = None
chart_sheet.add_chart(top_chart, "A22")

chart_sheet.merge_cells("A52:H53")
chart_sheet["A52"] = "阅读方式：先看各优先级覆盖金额，判断谈判候选的整体规模；再看Top 15，优先核实金额较高物料的需求增长原因并准备谈判。"
chart_sheet["A52"].fill = yellow_fill
chart_sheet["A52"].alignment = Alignment(vertical="center", wrap_text=True)
chart_sheet["A52"].font = Font(color="7F6000", size=10)

column_widths = {"A": 12, "B": 12, "C": 4, "D": 16, "E": 12, "F": 4, "G": 12, "H": 12, "J": 14, "K": 40, "L": 32, "M": 16, "N": 26}
for column, width in column_widths.items():
    chart_sheet.column_dimensions[column].width = width

workbook.save(report_path)
print("已导出报告、月度数据、规则说明和图表工作表。")